# Sesión 3: Árboles de Decisión
## Predicción con Reglas Interpretables

**Objetivo**: Aprender cómo funcionan los árboles de decisión y compararlos con la regresión logística.

**Duración**: ~60 minutos

**Pregunta**: ¿Podemos crear un modelo que tome decisiones usando reglas simples tipo "si-entonces"?

## 1. Configuración y Carga de Datos

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Librerías de machine learning
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# Cargar dataset
data_path = Path('../data/datos_liga_futbol.csv')
df = pd.read_csv(data_path)

print(f"✓ Dataset cargado: {df.shape[0]} partidos")
print(f"✓ Librerías importadas")

## 2. Preprocesamiento (Feature Engineering)

Usaremos el mismo preprocesamiento que en la Sesión 2.

In [ ]:
# Crear features (mismo que Sesión 2)
df['Diferencia_Habilidad'] = df['Habilidad_Local'] - df['Habilidad_Visitante']
df['Diferencia_Racha'] = df['Racha_Local'] - df['Racha_Visitante']
df['Ratio_Habilidad'] = df['Habilidad_Local'] / df['Habilidad_Visitante']

# Seleccionar features
features = [
    'Habilidad_Local',
    'Habilidad_Visitante',
    'Racha_Local',
    'Racha_Visitante',
    'Diferencia_Habilidad',
    'Diferencia_Racha',
    'Ratio_Habilidad'
]

X = df[features]
y = df['Resultado']

# Split train/test (MISMO random_state para comparación justa)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("✓ Features creados")
print(f"\nTrain set: {len(X_train)} partidos")
print(f"Test set: {len(X_test)} partidos")

## 3. ¿Qué es un Árbol de Decisión?

### 🌳 Conceptos Básicos

Un **Árbol de Decisión** es un modelo que toma decisiones mediante una serie de preguntas tipo "sí/no".

**Analogía**: Imagina que eres un experto en fútbol y alguien te pregunta quién ganará un partido:

```
¿El equipo local tiene más habilidad?
├─ SÍ (Diferencia_Habilidad > 5)
│  └─ ¿El local tiene racha positiva?
│     ├─ SÍ → Predecir: VICTORIA LOCAL (confianza alta)
│     └─ NO → Predecir: VICTORIA LOCAL (confianza media)
└─ NO (Diferencia_Habilidad ≤ 5)
   └─ ¿Los equipos están muy igualados?
      ├─ SÍ → Predecir: EMPATE
      └─ NO → Predecir: VICTORIA VISITANTE
```

### 💡 Ventajas vs Regresión Logística

**Árbol de Decisión**:
- ✓ Fácil de interpretar (reglas claras)
- ✓ No requiere normalización de datos
- ✓ Captura relaciones no lineales
- ✗ Puede sobreajustarse (overfitting)

**Regresión Logística**:
- ✓ Más estable, generaliza mejor
- ✗ Menos interpretable
- ✗ Solo captura relaciones lineales

## 4. Entrenamiento del Árbol de Decisión

In [ ]:
# Crear y entrenar el árbol
arbol = DecisionTreeClassifier(
    max_depth=5,  # Profundidad máxima (evitar overfitting)
    min_samples_split=10,  # Mínimo de muestras para dividir un nodo
    min_samples_leaf=5,  # Mínimo de muestras en cada hoja
    random_state=42
)

# Entrenar
arbol.fit(X_train, y_train)

print("✓ Árbol de Decisión entrenado")
print(f"\nProfundidad del árbol: {arbol.get_depth()}")
print(f"Número de hojas: {arbol.get_n_leaves()}")
print(f"Número de nodos: {arbol.tree_.node_count}")

## 5. Visualización del Árbol

Veamos cómo el árbol toma decisiones.

In [ ]:
# Visualizar el árbol completo
plt.figure(figsize=(20, 10))
plot_tree(arbol, 
          feature_names=features,
          class_names=arbol.classes_,
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Árbol de Decisión Completo', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n📚 CÓMO LEER EL ÁRBOL:")
print("\nCada nodo muestra:")
print("  1. Regla de decisión (ej: 'Diferencia_Habilidad <= 2.5')")
print("  2. gini: Medida de impureza (0 = puro, 0.5 = muy mezclado)")
print("  3. samples: Cantidad de partidos que llegan a este nodo")
print("  4. value: [Victoria Local, Empate, Victoria Visitante]")
print("  5. class: Clase predicha (la mayoritaria)")
print("\nColor:")
print("  - Más intenso = Mayor confianza en la predicción")
print("  - Naranja: Victoria Local")
print("  - Azul: Victoria Visitante")
print("  - Verde: Empate")

## 6. Primeros 3 Niveles del Árbol (Vista Simplificada)

Para entender mejor las reglas principales, veamos solo los primeros niveles.

In [ ]:
# Entrenar un árbol más simple para visualización
arbol_simple = DecisionTreeClassifier(
    max_depth=3,
    min_samples_split=10,
    random_state=42
)
arbol_simple.fit(X_train, y_train)

# Visualizar
plt.figure(figsize=(18, 8))
plot_tree(arbol_simple, 
          feature_names=features,
          class_names=arbol_simple.classes_,
          filled=True,
          rounded=True,
          fontsize=12)
plt.title('Árbol Simplificado (3 niveles) - Reglas Principales', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 Esta versión simplificada muestra las decisiones MÁS IMPORTANTES.")

## 7. Evaluación del Modelo

In [ ]:
# Predicciones
y_train_pred = arbol.predict(X_train)
y_test_pred = arbol.predict(X_test)

# Accuracy
train_accuracy_tree = accuracy_score(y_train, y_train_pred)
test_accuracy_tree = accuracy_score(y_test, y_test_pred)

print("RESULTADOS - ÁRBOL DE DECISIÓN:")
print(f"\nAccuracy en TRAIN: {train_accuracy_tree:.3f} ({train_accuracy_tree*100:.1f}%)")
print(f"Accuracy en TEST:  {test_accuracy_tree:.3f} ({test_accuracy_tree*100:.1f}%)")

# Matriz de confusión
cm = confusion_matrix(y_test, y_test_pred, labels=arbol.classes_)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=arbol.classes_, 
            yticklabels=arbol.classes_,
            cbar_kws={'label': 'Cantidad de partidos'})
plt.title('Matriz de Confusión - Árbol de Decisión', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Predicción del Modelo', fontsize=12)
plt.ylabel('Resultado Real', fontsize=12)
plt.tight_layout()
plt.show()

# Classification report
print("\nREPORTE DETALLADO:")
print("="*60)
print(classification_report(y_test, y_test_pred, target_names=arbol.classes_))

## 8. Comparación con Regresión Logística

Entrenemos también una regresión logística para comparar.

In [ ]:
# Entrenar regresión logística (mismo train/test)
from sklearn.linear_model import LogisticRegression

modelo_logistico = LogisticRegression(max_iter=1000, random_state=42)
modelo_logistico.fit(X_train, y_train)

# Predicciones
y_test_pred_log = modelo_logistico.predict(X_test)
test_accuracy_log = accuracy_score(y_test, y_test_pred_log)

# Tabla comparativa
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Árbol de Decisión'],
    'Accuracy Test': [test_accuracy_log, test_accuracy_tree],
    'Accuracy Train': [accuracy_score(y_train, modelo_logistico.predict(X_train)), train_accuracy_tree]
})

# Calcular diferencia train-test (indicador de overfitting)
comparacion['Diferencia Train-Test'] = comparacion['Accuracy Train'] - comparacion['Accuracy Test']

print("COMPARACIÓN DE MODELOS:")
print("="*70)
print(comparacion.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Accuracy comparativo
x = np.arange(len(comparacion))
width = 0.35
axes[0].bar(x - width/2, comparacion['Accuracy Train'], width, label='Train', alpha=0.8)
axes[0].bar(x + width/2, comparacion['Accuracy Test'], width, label='Test', alpha=0.8)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy: Train vs Test', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparacion['Modelo'])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1])

# Gráfico 2: Diferencia train-test (overfitting)
colors = ['green' if x < 0.1 else 'orange' for x in comparacion['Diferencia Train-Test']]
axes[1].bar(comparacion['Modelo'], comparacion['Diferencia Train-Test'], color=colors, alpha=0.7)
axes[1].set_ylabel('Diferencia (Train - Test)', fontsize=12)
axes[1].set_title('Indicador de Overfitting', fontsize=14, fontweight='bold')
axes[1].axhline(0.1, color='red', linestyle='--', label='Umbral razonable')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 INTERPRETACIÓN:")
if test_accuracy_tree > test_accuracy_log:
    print(f"  ✓ El Árbol de Decisión supera a la Regresión Logística")
    print(f"    Mejora: +{(test_accuracy_tree - test_accuracy_log)*100:.1f} puntos porcentuales")
elif test_accuracy_tree < test_accuracy_log:
    print(f"  ✓ La Regresión Logística supera al Árbol de Decisión")
    print(f"    Diferencia: {(test_accuracy_log - test_accuracy_tree)*100:.1f} puntos porcentuales")
else:
    print(f"  ≈ Ambos modelos tienen desempeño similar")

# Análisis de overfitting
if comparacion.loc[1, 'Diferencia Train-Test'] > 0.1:
    print(f"\n  ⚠️  El árbol muestra signos de overfitting")
    print(f"     Diferencia train-test: {comparacion.loc[1, 'Diferencia Train-Test']:.3f}")
else:
    print(f"\n  ✓ El árbol generaliza bien (bajo overfitting)")

## 9. Feature Importance (Importancia de Variables)

¿Qué variables usa más el árbol para tomar decisiones?

In [ ]:
# Obtener importancias
importances = pd.DataFrame({
    'Feature': features,
    'Importancia': arbol.feature_importances_
}).sort_values('Importancia', ascending=False)

print("IMPORTANCIA DE FEATURES:")
print("="*50)
print(importances.to_string(index=False))
print(f"\nTotal: {importances['Importancia'].sum():.3f} (debe sumar 1.0)")

# Visualización
plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(importances['Importancia'] / importances['Importancia'].max())
plt.barh(importances['Feature'], importances['Importancia'], color=colors)
plt.xlabel('Importancia', fontsize=12)
plt.title('Importancia de Features en el Árbol de Decisión', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 INTERPRETACIÓN:")
top_feature = importances.iloc[0]
print(f"  - La variable más importante es: {top_feature['Feature']}")
print(f"    Contribuye con {top_feature['Importancia']*100:.1f}% a las decisiones del árbol")
print(f"\n  - Top 3 features:")
for i in range(min(3, len(importances))):
    print(f"    {i+1}. {importances.iloc[i]['Feature']}: {importances.iloc[i]['Importancia']*100:.1f}%")

## 10. Ejemplos de Predicciones con Explicación

Veamos cómo el árbol toma decisiones para casos específicos.

In [ ]:
# Obtener la ruta de decisión para un ejemplo
def explicar_decision(modelo, X_ejemplo, feature_names):
    """Explica la ruta de decisión del árbol para un ejemplo."""
    decision_path = modelo.decision_path(X_ejemplo)
    leaf_id = modelo.apply(X_ejemplo)
    
    feature = modelo.tree_.feature
    threshold = modelo.tree_.threshold
    
    nodo_ids = decision_path.indices[decision_path.indptr[0]:decision_path.indptr[1]]
    
    print("\n  📍 RUTA DE DECISIÓN:")
    for i, nodo_id in enumerate(nodo_ids[:-1]):  # Excluir la hoja
        if feature[nodo_id] != -2:  # No es hoja
            feature_name = feature_names[feature[nodo_id]]
            threshold_val = threshold[nodo_id]
            valor_ejemplo = X_ejemplo.iloc[0, feature[nodo_id]]
            
            if valor_ejemplo <= threshold_val:
                direccion = "≤"
            else:
                direccion = ">"
            
            print(f"     Nivel {i+1}: {feature_name} = {valor_ejemplo:.2f} {direccion} {threshold_val:.2f}")

# Seleccionar 3 ejemplos
ejemplos_indices = [10, 50, 100]  # Índices fijos para reproducibilidad
ejemplos = df.loc[ejemplos_indices]

print("EJEMPLOS DE PREDICCIONES CON EXPLICACIÓN:")
print("="*80)

for i, idx in enumerate(ejemplos_indices, 1):
    partido = ejemplos.loc[idx]
    X_ejemplo = partido[features].to_frame().T
    
    pred = arbol.predict(X_ejemplo)[0]
    probs = arbol.predict_proba(X_ejemplo)[0]
    
    print(f"\n📊 PARTIDO {i}:")
    print(f"   {partido['Equipo_Local']} vs {partido['Equipo_Visitante']}")
    print(f"   Habilidades: {partido['Habilidad_Local']} vs {partido['Habilidad_Visitante']}")
    print(f"   Rachas: {partido['Racha_Local']} vs {partido['Racha_Visitante']}")
    
    explicar_decision(arbol, X_ejemplo, features)
    
    print(f"\n  🎯 RESULTADO:")
    print(f"     Real: {partido['Resultado']}")
    print(f"     Predicción: {pred}")
    print(f"     {'✓ CORRECTO' if pred == partido['Resultado'] else '✗ INCORRECTO'}")
    
    print(f"\n  📊 PROBABILIDADES:")
    for j, clase in enumerate(arbol.classes_):
        print(f"     {clase}: {probs[j]*100:.1f}%")
    print("-" * 80)

## 11. Resumen y Conclusiones

### 🌳 ¿Qué Aprendimos?

1. **Los árboles de decisión**:
   - Toman decisiones mediante reglas "si-entonces"
   - Son fáciles de interpretar y visualizar
   - Pueden capturar relaciones no lineales
   - Riesgo de overfitting si no se controla su profundidad

2. **Hiperparámetros importantes**:
   - `max_depth`: Profundidad máxima del árbol
   - `min_samples_split`: Mínimo de muestras para dividir
   - `min_samples_leaf`: Mínimo de muestras en hojas

3. **Comparación con Regresión Logística**:
   - Ambos modelos tienen fortalezas y debilidades
   - Árbol: más interpretable, puede sobreajustarse
   - Logística: más estable, menos interpretable

### 📊 Métricas Guardadas

Guardaremos las métricas para comparación final.

In [ ]:
# Guardar métricas
metricas_arbol = {
    'Modelo': 'Árbol de Decisión',
    'Accuracy_Train': train_accuracy_tree,
    'Accuracy_Test': test_accuracy_tree,
    'Profundidad': arbol.get_depth(),
    'Num_Hojas': arbol.get_n_leaves()
}

# Tabla comparativa final
comparacion_final = pd.DataFrame([
    {
        'Modelo': 'Regresión Logística',
        'Accuracy Test': f"{test_accuracy_log:.3f}",
        'Interpretabilidad': 'Media',
        'Overfitting': 'Bajo'
    },
    {
        'Modelo': 'Árbol de Decisión',
        'Accuracy Test': f"{test_accuracy_tree:.3f}",
        'Interpretabilidad': 'Alta',
        'Overfitting': 'Medio' if (train_accuracy_tree - test_accuracy_tree) > 0.1 else 'Bajo'
    }
])

print("="*70)
print("RESUMEN FINAL - COMPARACIÓN DE MODELOS")
print("="*70)
print(comparacion_final.to_string(index=False))

print(f"\n💾 Métricas guardadas para comparación futura")
print(f"\n🎯 PRÓXIMO PASO: Sesión 4 - Random Forest")
print("   ¿Qué pasa si combinamos MUCHOS árboles?")
print("   Spoiler: Ensemble Learning - ¡La unión hace la fuerza!")
print("="*70)